In [ ]:
import sys
import os
from pathlib import Path  # noqa: F401

# Resolve project root regardless of where the notebook is launched from
for _candidate in [".", "..", "../.."]:
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import matplotlib.pyplot as plt  # noqa: E402
import seaborn as sns  # noqa: E402
import mne  # noqa: E402

from src.io.parsing import DatasetParser  # noqa: E402
from src.definitions.fields import (  # noqa: E402
    ExperimentNames,
    SingleDataMetadata,
)
from src.definitions.constants import ProjectPaths  # noqa: E402

mne.set_log_level("ERROR")
%matplotlib inline
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.0)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)
print("Setup complete.")

# ASSR Data Inspection — Annotations

This notebook inspects the **annotations** of every recording in the **ASSR** raw dataset.

Goals:
* Load the annotations of *all* ASSR `.edf` recordings.
* Report **which annotation labels** appear and **how many of each** per recording.
* Inspect annotation **onsets** and the **distribution of inter-onset intervals**
  (the gap, in seconds, between each consecutive annotation) — both across all
  labels and for the main stimulus label.

> Annotations are read straight from the EDF headers (`preload=False`), so this is
> fast and does not load the full multi-channel signal into memory.

## Configuration

In [ ]:
EXPERIMENT = ExperimentNames.ASSR

# Main stimulus label of interest for the per-label onset analysis. The ASSR
# recordings use "fam+" as the per-trial stimulus marker (see
# src/preprocessing/stimulus_alignment.py -> DEFAULT_STIMULUS_LABEL).
STIMULUS_LABEL = "fam+"

# ── Plot saving ──────────────────────────────────────────────────────────────
SAVE_PLOTS = True
PLOTS_DIR = (
    ProjectPaths.NOTEBOOKS_DIR / "00-preprocessing" / "plots" / "assr_data_inspection"
)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Plots will be saved to: {PLOTS_DIR}")

## Dataset Discovery

In [ ]:
raw_data_dir, participant_map_path = ProjectPaths.get_experiment_data_dir(
    EXPERIMENT, is_processed=False
)
parser = DatasetParser(EXPERIMENT, participant_map_path)
metadata_df = parser.parse_dataset_filenames(raw_data_dir)

# Make the enum-keyed metadata easier to work with.
meta = pd.DataFrame(
    {
        "participant_id": metadata_df[SingleDataMetadata.PARTICIPANT_ID],
        "eeg_id": metadata_df[SingleDataMetadata.EEG_CONDITION_ID].apply(
            lambda e: e.value
        ),
        "condition": metadata_df[SingleDataMetadata.CONDITION].apply(lambda e: e.value),
        "filename": metadata_df[SingleDataMetadata.FILENAME],
    }
).sort_values(["participant_id", "eeg_id", "filename"]).reset_index(drop=True)

print(f"Raw data dir: {raw_data_dir}")
print(f"ASSR recordings discovered: {len(meta)}")
print(f"Participants: {meta['participant_id'].nunique()}")
print(meta["condition"].value_counts().to_string())
meta.head()

## Load Annotations From All Recordings

In [ ]:
records = []  # one row per annotation (long / tidy format)
load_errors = []

for row in meta.itertuples(index=False):
    fpath = raw_data_dir / row.filename
    try:
        raw = mne.io.read_raw_edf(fpath, preload=False, verbose="ERROR")
    except Exception as exc:  # noqa: BLE001
        load_errors.append((row.filename, repr(exc)))
        continue

    ann = raw.annotations
    if len(ann) == 0:
        load_errors.append((row.filename, "no annotations"))
        continue

    for onset, duration, description in zip(ann.onset, ann.duration, ann.description):
        records.append(
            {
                "filename": row.filename,
                "participant_id": row.participant_id,
                "eeg_id": row.eeg_id,
                "condition": row.condition,
                "label": description,
                "onset": float(onset),
                "duration": float(duration),
            }
        )

annotations_df = pd.DataFrame(records)

print(f"Recordings successfully read: {annotations_df['filename'].nunique()} / {len(meta)}")
print(f"Total annotations loaded: {len(annotations_df)}")
if load_errors:
    print(f"\n{len(load_errors)} recording(s) skipped / problematic:")
    for fn, msg in load_errors:
        print(f"  - {fn}: {msg}")
annotations_df.head(10)

## Annotation Labels — Overall Distribution

In [ ]:
# Which labels exist across the whole ASSR dataset, how often in total, and in how
# many recordings each appears.
label_overview = (
    annotations_df.groupby("label")
    .agg(
        total_count=("label", "size"),
        n_recordings=("filename", "nunique"),
        mean_per_recording=("filename", lambda s: s.size / s.nunique()),
    )
    .sort_values("total_count", ascending=False)
)
label_overview["n_recordings_pct"] = (
    100 * label_overview["n_recordings"] / annotations_df["filename"].nunique()
).round(1)
print(f"Unique annotation labels: {len(label_overview)}")
label_overview

In [ ]:
fig, ax = plt.subplots(figsize=(9, max(3, 0.5 * len(label_overview))))
sns.barplot(
    data=label_overview.reset_index(),
    y="label",
    x="total_count",
    color="steelblue",
    ax=ax,
)
ax.set_title("Total annotation count per label (all ASSR recordings)")
ax.set_xlabel("Total count across dataset")
ax.set_ylabel("Annotation label")
for container in ax.containers:
    ax.bar_label(container, fmt="%d", padding=3)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "label_total_counts.png", dpi=150, bbox_inches="tight")
plt.show()

## Annotation Counts Per Recording

In [ ]:
# Count of each label within every recording (rows = recordings, cols = labels).
counts_per_file = (
    annotations_df.groupby(["filename", "label"]).size().unstack(fill_value=0)
)
# Order columns by overall frequency for readability.
counts_per_file = counts_per_file[label_overview.index]
counts_per_file["TOTAL"] = counts_per_file.sum(axis=1)

# Attach participant / condition metadata for context.
counts_per_file = (
    meta.set_index("filename")[["participant_id", "eeg_id", "condition"]]
    .join(counts_per_file, how="right")
    .sort_values(["participant_id", "eeg_id"])
)
print(f"Recordings with annotations: {len(counts_per_file)}")
counts_per_file.head()

In [ ]:
# Quick consistency check: do recordings agree on label counts, or do they vary?
label_count_cols = [c for c in label_overview.index]
summary_across_files = counts_per_file[label_count_cols + ["TOTAL"]].describe().T[
    ["mean", "std", "min", "max"]
]
summary_across_files["n_distinct_values"] = [
    counts_per_file[c].nunique() for c in label_count_cols + ["TOTAL"]
]
print("Per-label count statistics across recordings:")
summary_across_files

# Stimulus Onset Differences (fam+)

In [ ]:
# IOIs for the main stimulus label only.
stim_df = annotations_df[annotations_df["label"] == STIMULUS_LABEL]
if stim_df.empty:
    print(
        f"WARNING: stimulus label {STIMULUS_LABEL!r} not found. "
        f"Available labels: {sorted(annotations_df['label'].unique())}"
    )
    ioi_stim = pd.DataFrame(columns=["filename", "ioi"])
else:
    ioi_stim = inter_onset_intervals(stim_df)
    ioi_stim = ioi_stim.merge(
        meta[["filename", "participant_id", "condition"]], on="filename", how="left"
    )
    print(f"Inter-onset intervals for stimulus label {STIMULUS_LABEL!r}:")
    print(ioi_stim["ioi"].describe().to_string())
    print(
        f"\nRange (min..max gap): {ioi_stim['ioi'].min():.4f}s .. "
        f"{ioi_stim['ioi'].max():.4f}s"
    )

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(7, 5))

sns.histplot(ioi_stim["ioi"], bins=60, color="teal", ax=axes)
axes.set_title(f"Inter-onset intervals — {STIMULUS_LABEL!r}")
axes.set_xlabel("Gap to next stimulus (s)")
axes.axvline(ioi_stim["ioi"].median(), color="crimson", ls="--",
                label=f"median={ioi_stim['ioi'].median():.3f}s")
axes.legend()

fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "inter_onset_interval_histograms.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Per-recording onset-timing summary: how many onsets, total span, and the
# inter-onset-interval range/mean within each recording.
def per_file_onset_summary(df: pd.DataFrame, label: str) -> pd.DataFrame:
    out = []
    for filename, grp in df.groupby("filename"):
        onsets = np.sort(grp["onset"].to_numpy())
        diffs = np.diff(onsets) if onsets.size >= 2 else np.array([])
        out.append(
            {
                "filename": filename,
                "label_scope": label,
                "n_onsets": onsets.size,
                "first_onset_s": onsets.min() if onsets.size else np.nan,
                "last_onset_s": onsets.max() if onsets.size else np.nan,
                "span_s": (onsets.max() - onsets.min()) if onsets.size else np.nan,
                "ioi_min_s": diffs.min() if diffs.size else np.nan,
                "ioi_max_s": diffs.max() if diffs.size else np.nan,
                "ioi_range_s": (diffs.max() - diffs.min()) if diffs.size else np.nan,
                "ioi_mean_s": diffs.mean() if diffs.size else np.nan,
                "ioi_std_s": diffs.std() if diffs.size else np.nan,
            }
        )
    return pd.DataFrame(out)


onset_summary_all = per_file_onset_summary(annotations_df, "all_labels")
onset_summary_stim = (
    per_file_onset_summary(stim_df, STIMULUS_LABEL) if not stim_df.empty else pd.DataFrame()
)

onset_summary = pd.concat([onset_summary_all, onset_summary_stim], ignore_index=True)
onset_summary = onset_summary.merge(
    meta[["filename", "participant_id", "condition"]], on="filename", how="left"
).round(3)
onset_summary.sort_values(["label_scope", "participant_id"])

onset_summary.head()

In [ ]:
# Distribution of inter-onset intervals per recording for the stimulus label
# (or all labels if the stimulus label is absent).
box_source = ioi_stim if not ioi_stim.empty else ioi_all
scope = STIMULUS_LABEL if not ioi_stim.empty else "all labels"

order = sorted(box_source["filename"].unique())
fig, ax = plt.subplots(figsize=(12, max(6, 0.3 * len(order))))
sns.boxplot(
    data=box_source,
    y="filename",
    x="ioi",
    order=order,
    color="lightsteelblue",
    fliersize=2,
    ax=ax,
)
ax.set_title(f"Inter-onset interval distribution per recording — {scope}")
ax.set_xlabel("Gap to next onset (s)")
ax.set_ylabel("Recording (filename)")
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "ioi_per_recording_boxplot.png", dpi=150, bbox_inches="tight")
plt.show()

## `bgin` vs `fam+` Comparison

The ASSR event stream is built from two markers that occur as pairs: a `bgin`
("begin") marker immediately followed (~2 ms later) by a `fam+` stimulus marker.
Here we compare them directly:

* **consecutive same-label intervals** — gap between successive `fam+` markers, and
  between successive `bgin` markers;
* **within-pair gap** — `bgin` → the `fam+` that immediately follows it;
* **special cases** — any `bgin` *not* immediately followed by a `fam+` (e.g. two
  `bgin` back-to-back with no `fam+` between them), reported with separate statistics.

In [ ]:
# Build the bgin/fam+ sequence structures per recording.
PAIR_LABELS = ["bgin", STIMULUS_LABEL]

consecutive_rows = []   # consecutive same-label intervals (bgin->bgin, fam+->fam+)
pair_rows = []          # bgin -> the fam+ that immediately follows it
orphan_bgin_rows = []   # bgin NOT immediately followed by a fam+

pair_subset = annotations_df[annotations_df["label"].isin(PAIR_LABELS)]
for filename, grp in pair_subset.groupby("filename"):
    g = grp.sort_values("onset").reset_index(drop=True)
    labels = g["label"].to_numpy()
    onsets = g["onset"].to_numpy()
    n = len(g)

    # Consecutive intervals within each label.
    for lab in PAIR_LABELS:
        sel = np.sort(onsets[labels == lab])
        for d in np.diff(sel):
            consecutive_rows.append(
                {"filename": filename, "label": lab, "interval_s": float(d)}
            )

    # Walk the sequence: pair each bgin with the following fam+, else flag it.
    for i in range(n):
        if labels[i] != "bgin":
            continue
        nxt_label = labels[i + 1] if i + 1 < n else None
        if nxt_label == STIMULUS_LABEL:
            pair_rows.append(
                {
                    "filename": filename,
                    "bgin_onset_s": float(onsets[i]),
                    "fam_onset_s": float(onsets[i + 1]),
                    "pair_gap_s": float(onsets[i + 1] - onsets[i]),
                }
            )
        else:
            orphan_bgin_rows.append(
                {
                    "filename": filename,
                    "seq_index": i,
                    "bgin_onset_s": float(onsets[i]),
                    "next_label": nxt_label if nxt_label is not None
                    else "<end-of-recording>",
                    "gap_to_next_s": float(onsets[i + 1] - onsets[i])
                    if i + 1 < n
                    else np.nan,
                }
            )

consecutive_df = pd.DataFrame(consecutive_rows)
pair_df = pd.DataFrame(pair_rows).merge(
    meta[["filename", "participant_id", "condition"]], on="filename", how="left"
)
orphan_bgin_df = pd.DataFrame(orphan_bgin_rows)
if not orphan_bgin_df.empty:
    orphan_bgin_df = orphan_bgin_df.merge(
        meta[["filename", "participant_id", "condition"]], on="filename", how="left"
    )

print(f"Consecutive same-label intervals: {len(consecutive_df)}")
print(f"Valid bgin -> fam+ pairs:         {len(pair_df)}")
print(f"Orphan bgin (no following fam+):  {len(orphan_bgin_df)}")

### Consecutive same-label intervals — `bgin` vs `fam+`

In [ ]:
# Side-by-side statistics for the gap between successive markers of each label.
consec_stats = consecutive_df.groupby("label")["interval_s"].describe()[
    ["count", "mean", "std", "min", "25%", "50%", "75%", "max"]
]
consec_stats["range_s"] = (
    consecutive_df.groupby("label")["interval_s"].max()
    - consecutive_df.groupby("label")["interval_s"].min()
)
consec_stats.round(4)

### Within-pair gap — `bgin` &rarr; following `fam+`

In [ ]:
print("bgin -> fam+ within-pair gap statistics (s):")
print(pair_df["pair_gap_s"].describe().to_string())
print(
    f"\nRange (min..max): {pair_df['pair_gap_s'].min():.4f}s .. "
    f"{pair_df['pair_gap_s'].max():.4f}s"
)
print("\nPer-recording within-pair gap summary:")
pair_per_file = (
    pair_df.groupby(["filename", "participant_id", "condition"])["pair_gap_s"]
    .agg(["count", "mean", "std", "min", "max"])
    .round(4)
)
pair_per_file.head()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: overlaid consecutive same-label intervals.
for lab, color in zip(PAIR_LABELS, ["darkorange", "teal"]):
    sub = consecutive_df.loc[consecutive_df["label"] == lab, "interval_s"]
    sns.histplot(
        sub, bins=60, color=color, label=lab, stat="density", alpha=0.5, ax=axes[0]
    )
axes[0].set_title("Consecutive same-label interval")
axes[0].set_xlabel("Gap to next same-label marker (s)")
axes[0].legend()

# Panel 2: within-pair gap (bgin -> fam+).
sns.histplot(pair_df["pair_gap_s"] * 1000, bins=60, color="purple", ax=axes[1])
axes[1].set_title("Within-pair gap: bgin -> fam+")
axes[1].set_xlabel("Gap (ms)")
axes[1].axvline(
    pair_df["pair_gap_s"].median() * 1000,
    color="crimson",
    ls="--",
    label=f"median={pair_df['pair_gap_s'].median() * 1000:.2f} ms",
)
axes[1].legend()

# Panel 3: boxplot comparison of consecutive intervals by label.
sns.boxplot(
    data=consecutive_df, x="label", y="interval_s", order=PAIR_LABELS,
    palette=["darkorange", "teal"], fliersize=2, ax=axes[2],
)
axes[2].set_title("Consecutive same-label interval by marker")
axes[2].set_xlabel("Marker label")
axes[2].set_ylabel("Interval (s)")

fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "bgin_vs_fam_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

### Special cases — `bgin` not followed by `fam+`

Any `bgin` whose next marker is *not* a `fam+` is flagged below. This includes
back-to-back `bgin` (two `bgin` with no `fam+` between them) and a trailing `bgin`
at the very end of a recording. Back-to-back `bgin` get their own statistics.

In [ ]:
if orphan_bgin_df.empty:
    print("No orphan bgin found — every bgin is immediately followed by a fam+.")
else:
    print(
        f"Found {len(orphan_bgin_df)} orphan bgin across "
        f"{orphan_bgin_df['filename'].nunique()} recording(s).\n"
    )
    print("What follows each orphan bgin:")
    print(orphan_bgin_df["next_label"].value_counts().to_string())
    print("\nOrphan bgin count per recording:")
    print(
        orphan_bgin_df.groupby(["filename", "participant_id", "condition"])
        .size()
        .rename("n_orphan_bgin")
        .to_string()
    )
orphan_bgin_df

In [ ]:
# Separate statistics for the back-to-back case: a bgin directly followed by
# another bgin (no fam+ in between).
if orphan_bgin_df.empty:
    consecutive_bgin = orphan_bgin_df
else:
    consecutive_bgin = orphan_bgin_df[orphan_bgin_df["next_label"] == "bgin"]

if consecutive_bgin.empty:
    print("No back-to-back bgin (bgin directly followed by another bgin) detected.")
else:
    print(
        f"Back-to-back bgin cases: {len(consecutive_bgin)} across "
        f"{consecutive_bgin['filename'].nunique()} recording(s).\n"
    )
    print("Interval (s) between such consecutive bgin:")
    print(consecutive_bgin["gap_to_next_s"].describe().to_string())
    print(
        f"\nRange: {consecutive_bgin['gap_to_next_s'].min():.4f}s .. "
        f"{consecutive_bgin['gap_to_next_s'].max():.4f}s"
    )
consecutive_bgin

### `bgin` consecutive intervals — with vs without orphan pairs

The `bgin`&rarr;`bgin` interval statistics above include the short intervals that
start at an **orphan** `bgin` (a `bgin` with no `fam+`, immediately followed by the
next `bgin`). Below we report the `bgin` interval distribution both **with** those
orphan pairs and **without** them (excluding every interval whose *starting* `bgin`
is an orphan), so the orphan back-to-back gaps do not bias the regular-cadence
statistics.

In [ ]:
# Onsets of orphan bgin per recording (the bgin that starts an orphan pair).
orphan_onsets_by_file = (
    orphan_bgin_df.groupby("filename")["bgin_onset_s"].apply(set).to_dict()
    if not orphan_bgin_df.empty
    else {}
)

bgin_interval_rows = []
bgin_only = annotations_df[annotations_df["label"] == "bgin"]
for filename, grp in bgin_only.groupby("filename"):
    onsets = np.sort(grp["onset"].to_numpy())
    orphans = orphan_onsets_by_file.get(filename, set())
    for a, b in zip(onsets[:-1], onsets[1:]):
        # The interval "is an orphan pair" when it starts at an orphan bgin
        # (i.e. it is the back-to-back gap orphan_bgin -> next bgin).
        bgin_interval_rows.append(
            {
                "filename": filename,
                "interval_s": float(b - a),
                "is_orphan_pair": float(a) in orphans,
            }
        )

bgin_intervals_df = pd.DataFrame(bgin_interval_rows)


def _describe(s: pd.Series) -> pd.Series:
    return s.describe()[
        ["count", "mean", "std", "min", "25%", "50%", "75%", "max"]
    ]


clean_bgin = bgin_intervals_df.loc[~bgin_intervals_df["is_orphan_pair"], "interval_s"]
bgin_orphan_compare = pd.DataFrame(
    {
        "with_orphan_pairs": _describe(bgin_intervals_df["interval_s"]),
        "without_orphan_pairs": _describe(clean_bgin),
    }
).T
bgin_orphan_compare["range_s"] = (
    bgin_orphan_compare["max"] - bgin_orphan_compare["min"]
)

print(f"Total bgin->bgin intervals: {len(bgin_intervals_df)}")
print(f"Orphan-pair intervals excluded: {int(bgin_intervals_df['is_orphan_pair'].sum())}")
bgin_orphan_compare.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(
    bgin_intervals_df["interval_s"],
    bins=60,
    color="lightgray",
    label=f"with orphan pairs (n={len(bgin_intervals_df)})",
    stat="density",
    ax=ax,
)
sns.histplot(
    clean_bgin,
    bins=60,
    color="darkorange",
    label=f"without orphan pairs (n={len(clean_bgin)})",
    stat="density",
    alpha=0.6,
    ax=ax,
)
ax.axvline(
    bgin_intervals_df["interval_s"].mean(),
    color="gray", ls="--",
    label=f"mean w/ ={bgin_intervals_df['interval_s'].mean():.4f}s",
)
ax.axvline(
    clean_bgin.mean(),
    color="darkorange", ls="--",
    label=f"mean w/o={clean_bgin.mean():.4f}s",
)
ax.set_title("bgin consecutive interval — with vs without orphan pairs")
ax.set_xlabel("Interval to next bgin (s)")
ax.legend()
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(
        PLOTS_DIR / "bgin_interval_with_without_orphan.png",
        dpi=150,
        bbox_inches="tight",
    )
plt.show()

## Export Summary Tables (optional)

In [ ]:
if SAVE_PLOTS:
    counts_per_file.to_csv(PLOTS_DIR / "label_counts_per_recording.csv")
    label_overview.to_csv(PLOTS_DIR / "label_overview.csv")
    onset_summary.to_csv(PLOTS_DIR / "onset_summary.csv", index=False)
    consec_stats.to_csv(PLOTS_DIR / "consecutive_interval_stats.csv")
    pair_df.to_csv(PLOTS_DIR / "bgin_fam_pairs.csv", index=False)
    orphan_bgin_df.to_csv(
        PLOTS_DIR / "orphan_bgin.csv", index=False
    )
    bgin_orphan_compare.to_csv(
        PLOTS_DIR / "bgin_interval_orphan_comparison.csv"
    )
    print(f"Summary CSVs written to: {PLOTS_DIR}")
else:
    print("SAVE_PLOTS is False — skipping CSV export.")